# Tensor Parallelism: Splitting One Layer's Matrix Across CPU + GPU

**My setup:** Intel i5 (CPU) + NVIDIA GTX 1650 (1 CUDA GPU).

This is different from Model Parallelism (splitting *whole layers* across devices) and Data
Parallelism (splitting *data* across full model copies). Here we split **a single layer's weight
matrix** into pieces (shards), give the **same input** to every device, and each device computes
its own piece of the same matrix multiplication — exactly the idea in `04-tensor-parallelism.md`.

**The layer we'll split:** Layer 1, a `Linear(4 -> 8)` layer.

- We cut its weight matrix into 2 column-shards: `Linear(4 -> 4)` on CPU, `Linear(4 -> 4)` on GPU.
- Both shards see the exact same input (4 features).
- CPU's shard computes 4 of the 8 output columns, GPU's shard computes the other 4.
- We then **combine** (concatenate) the two partial outputs into the full 8-wide output —
  mathematically identical to running one `Linear(4 -> 8)` layer on a single device.

Layer 2 (`Linear(8 -> 1)`) stays on the GPU as a normal, un-split layer, just to keep the whole
network runnable end to end.

> If you get a second CUDA GPU later, change `device_shard1` / `device_shard2` below to `cuda:0`
> and `cuda:1` — nothing else changes.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce GTX 1650


## Step 1: Pick the Two Shard Devices

**Keywords:** `shard` = one piece of a single weight matrix that has been split up.


In [2]:
device_shard1 = torch.device("cpu")
device_shard2 = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# Layer 2 (not split) just lives wherever the final combined output should end up computing
device_layer2 = device_shard2

print("Shard 1 of Layer 1 runs on:", device_shard1)
print("Shard 2 of Layer 1 runs on:", device_shard2)
print("Layer 2 (unsplit) runs on:", device_layer2)


Shard 1 of Layer 1 runs on: cpu
Shard 2 of Layer 1 runs on: cuda:0
Layer 2 (unsplit) runs on: cuda:0


## Step 2: Define the Tensor-Parallel Model

Layer 1 is built as **two smaller Linear layers** instead of one big one:

- `layer1_shard1`: `Linear(4, 4)` — computes output columns 0-3, on `device_shard1`
- `layer1_shard2`: `Linear(4, 4)` — computes output columns 4-7, on `device_shard2`

Both receive the **same input** `x`. Their outputs are concatenated (`torch.cat`) to form the
full 8-wide output, matching what a single `Linear(4, 8)` would have produced.


In [3]:
class TensorParallelANN(nn.Module):
    def __init__(self, device_shard1, device_shard2, device_layer2):
        super().__init__()
        self.device_shard1 = device_shard1
        self.device_shard2 = device_shard2
        self.device_layer2 = device_layer2

        # Layer 1, split into 2 column-shards (together they act like Linear(4, 8))
        self.layer1_shard1 = nn.Linear(4, 4).to(device_shard1)
        self.layer1_shard2 = nn.Linear(4, 4).to(device_shard2)
        self.relu = nn.ReLU()

        # Layer 2, kept whole (not split), lives on device_layer2
        self.layer2 = nn.Linear(8, 1).to(device_layer2)

    def forward(self, x):
        # --- Tensor-parallel step: same input sent to BOTH shards ---
        x_for_shard1 = x.to(self.device_shard1)
        x_for_shard2 = x.to(self.device_shard2)

        out_shard1 = self.layer1_shard1(x_for_shard1)   # computed on device_shard1 (CPU)
        out_shard2 = self.layer1_shard2(x_for_shard2)   # computed on device_shard2 (GPU), at the "same time"

        # --- Combine step: bring both partial outputs to one device, then concatenate ---
        out_shard1_on_layer2_device = out_shard1.to(self.device_layer2)
        out_shard2_on_layer2_device = out_shard2.to(self.device_layer2)

        combined = torch.cat([out_shard1_on_layer2_device, out_shard2_on_layer2_device], dim=1)
        combined = self.relu(combined)

        # Layer 2 runs normally on the combined, full-width output
        output = self.layer2(combined)
        return output

model = TensorParallelANN(device_shard1, device_shard2, device_layer2)
print(model)


TensorParallelANN(
  (layer1_shard1): Linear(in_features=4, out_features=4, bias=True)
  (layer1_shard2): Linear(in_features=4, out_features=4, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=8, out_features=1, bias=True)
)


## Step 3: Confirm the Split Actually Happened

Check that each shard's weights really live on the device we assigned.


In [4]:
print("Shard 1 weight device:", model.layer1_shard1.weight.device)
print("Shard 2 weight device:", model.layer1_shard2.weight.device)
print("Layer 2 weight device:", model.layer2.weight.device)

print("\nShard 1 weight shape:", model.layer1_shard1.weight.shape, "(4 in, 4 out)")
print("Shard 2 weight shape:", model.layer1_shard2.weight.shape, "(4 in, 4 out)")
print("Together they act like one Linear(4, 8) layer.")


Shard 1 weight device: cpu
Shard 2 weight device: cuda:0
Layer 2 weight device: cuda:0

Shard 1 weight shape: torch.Size([4, 4]) (4 in, 4 out)
Shard 2 weight shape: torch.Size([4, 4]) (4 in, 4 out)
Together they act like one Linear(4, 8) layer.


## Step 4: Dummy Data

Same toy regression setup as the earlier notebooks: 4 input features -> 1 output number.


In [5]:
torch.manual_seed(0)

num_samples = 256
X = torch.randn(num_samples, 4)
true_weights = torch.tensor([2.0, -1.0, 0.5, 3.0])
y = (X @ true_weights).unsqueeze(1) + 0.1 * torch.randn(num_samples, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: torch.Size([256, 4])
y shape: torch.Size([256, 1])


## Step 5: Training Loop

Nothing special here compared to a normal PyTorch training loop — the tensor-parallel split is
already handled inside `forward()`. PyTorch's autograd automatically tracks gradients correctly
across both shards and both devices, even though they live on CPU and GPU.


In [6]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()

    predictions = model(X)                       # forward pass, splits across CPU + GPU internally
    targets = y.to(device_layer2)                 # move targets to where the output lives

    loss = criterion(predictions, targets)
    loss.backward()                               # backward pass, flows back into both shards
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss.item():.4f}")


Epoch  20/200 | Loss: 8.5638
Epoch  40/200 | Loss: 2.4945
Epoch  60/200 | Loss: 0.4886
Epoch  80/200 | Loss: 0.2478
Epoch 100/200 | Loss: 0.1989
Epoch 120/200 | Loss: 0.1768
Epoch 140/200 | Loss: 0.1641
Epoch 160/200 | Loss: 0.1555
Epoch 180/200 | Loss: 0.1487
Epoch 200/200 | Loss: 0.1430


## Step 6: Quick Sanity Check on a New Sample


In [7]:
model.eval()
with torch.no_grad():
    sample = torch.tensor([[1.0, 0.5, -0.2, 2.0]])
    output = model(sample)
    print("Prediction:", output.item())
    print("Prediction was computed on device:", output.device)


Prediction: 6.843297481536865
Prediction was computed on device: cuda:0


## Step 7: Prove the Split Is Mathematically Equivalent to One Big Layer

Let's build a normal, un-split `Linear(4, 8)` layer, manually copy the two shards' weights into
its top half and bottom half, and confirm it gives the exact same output as our tensor-parallel
version. This is the same equivalence claim made in `04-tensor-parallelism.md`.


In [8]:
# Build a single, un-split Linear(4, 8) layer
combined_layer1 = nn.Linear(4, 8)

with torch.no_grad():
    # Top 4 output rows come from shard 1, bottom 4 from shard 2
    combined_layer1.weight[0:4, :] = model.layer1_shard1.weight.cpu()
    combined_layer1.weight[4:8, :] = model.layer1_shard2.weight.cpu()
    combined_layer1.bias[0:4] = model.layer1_shard1.bias.cpu()
    combined_layer1.bias[4:8] = model.layer1_shard2.bias.cpu()

sample = torch.tensor([[1.0, 0.5, -0.2, 2.0]])

# Output using the single combined layer (CPU-only, no splitting)
single_device_output = torch.relu(combined_layer1(sample))

# Output using our tensor-parallel shards directly
with torch.no_grad():
    out1 = model.layer1_shard1(sample.to(device_shard1)).to("cpu")
    out2 = model.layer1_shard2(sample.to(device_shard2)).to("cpu")
    tensor_parallel_output = torch.relu(torch.cat([out1, out2], dim=1))

print("Single-device output:      ", single_device_output)
print("Tensor-parallel output:    ", tensor_parallel_output)
print("Match:", torch.allclose(single_device_output, tensor_parallel_output, atol=1e-6))


Single-device output:       tensor([[3.2380, 0.0000, 0.1161, 1.2910, 0.0761, 0.0000, 0.6493, 1.9327]],
       grad_fn=<ReluBackward0>)
Tensor-parallel output:     tensor([[3.2380, 0.0000, 0.1161, 1.2910, 0.0761, 0.0000, 0.6493, 1.9327]])
Match: True


## What Just Happened (Recap)

1. Layer 1's weight matrix was **cut into 2 column-shards** — one on CPU, one on GPU.
2. Both shards received the **exact same input** at the same time.
3. Each shard computed its own piece of the output, independently.
4. We **concatenated** the two partial outputs to get the full result — mathematically identical
   to one big `Linear(4, 8)` layer, as proven in Step 7.
5. Layer 2 stayed whole, just to complete the network.

**In a real large model**, this is exactly how huge attention or fully-connected layers get split
across multiple GPUs (Megatron-LM popularized this "column-parallel then row-parallel" pattern).
The difference is scale: real tensor parallelism splits matrices with thousands of columns across
many GPUs connected by very fast links (like NVLink), not a tiny 4x8 matrix across CPU and GPU.

### If You Get a Second CUDA GPU Later

Just change:

```python
device_shard1 = torch.device("cuda:0")
device_shard2 = torch.device("cuda:1")
```

Nothing else in this notebook needs to change.
